# Module 12: Simple Linear Regression

**Case:** How is a Vienna hotel's room price associated with its distance from
the city centre?

We build the analysis in small, typeable steps. The aim is not merely to run
`ols`; it is to connect the business question, graph, regression line,
coefficient units, fitted values, and residuals.

Adapted from the hotel-price cases in Chapters 7–8 of *Data Analysis for
Business, Economics, and Policy*.

## 1. Load and define the analysis sample

Each row is one hotel offer. `price` is the room price in US dollars and
`distance_miles` is distance from the city centre. We begin with three- and
four-star hotels in Vienna so the comparison is reasonably focused.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [ ]:
hotels = pd.read_csv("data/raw/hotels_course.csv")
vienna = hotels.query("city == 'Vienna' and accommodation == 'Hotel'").copy()
vienna.shape

In [ ]:
vienna[["hotel_id", "price", "distance_miles", "stars", "rating"]].head()

**Before calculating:** Which direction should the association have? What
factors besides distance could make two hotel prices different? Those omitted
variables matter when we interpret the slope.

In [ ]:
vienna[["price", "distance_miles", "stars", "rating"]].describe().round(2)

## 2. See the pattern before fitting a model

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=vienna, x="distance_miles", y="price", alpha=0.65, ax=ax)
ax.set(title="Vienna hotel prices and distance", xlabel="Distance (miles)", ylabel="Price (USD)")
plt.show()

In [ ]:
vienna["distance_group"] = pd.cut(
    vienna["distance_miles"], bins=[0, 1, 2, 4, np.inf], right=False
)
group_means = vienna.groupby("distance_group", observed=True)["price"].agg(["mean", "count"])
group_means.round(1)

The grouped means describe the sample without imposing a straight line. The
regression line is a compact approximation: it selects the intercept and slope
that minimize the sum of squared residuals.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=vienna, x="distance_miles", y="price", scatter_kws={"alpha": 0.55}, ax=ax)
ax.set(title="Linear approximation", xlabel="Distance (miles)", ylabel="Price (USD)")
plt.show()

## 3. Estimate and interpret the regression

In [ ]:
simple_model = smf.ols("price ~ distance_miles", data=vienna).fit(cov_type="HC3")
simple_model.summary()

In [ ]:
coefficient_table = pd.DataFrame(
    {
        "estimate": simple_model.params,
        "robust_se": simple_model.bse,
        "ci_low": simple_model.conf_int()[0],
        "ci_high": simple_model.conf_int()[1],
    }
)
coefficient_table.round(2)

In [ ]:
intercept = simple_model.params["Intercept"]
slope = simple_model.params["distance_miles"]
print(f"Predicted price at the centre: ${intercept:.0f}")
print(f"Average price difference per additional mile: ${slope:.0f}")

The slope is an **association in dollars per mile**, not the causal effect of
moving a hotel. Location quality, star category, ratings, and other features
differ across hotels too.

## 4. Fitted values, residuals, and predictions

In [ ]:
vienna["fitted_price"] = simple_model.fittedvalues
vienna["residual"] = simple_model.resid
vienna[["price", "fitted_price", "residual"]].head().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=vienna, x="fitted_price", y="residual", alpha=0.65, ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set(title="Residuals reveal what the line misses", xlabel="Fitted price", ylabel="Residual")
plt.show()

In [ ]:
new_distances = pd.DataFrame({"distance_miles": [0.5, 2.0, 5.0]})
prediction = simple_model.get_prediction(new_distances).summary_frame(alpha=0.05)
new_distances.join(prediction).round(1)

In [ ]:
vienna.nlargest(5, "residual")[
    ["hotel_id", "price", "distance_miles", "fitted_price", "residual"]
].round(1)

**Checkpoint:** A large positive residual means the hotel is more expensive
than the distance-only model predicts. It does not by itself tell us whether
the hotel is overpriced; the model ignores hotel quality.

Continue to Module 13, where we add those features deliberately.

### Optional extension

Compare `price ~ distance_miles` with `log_price ~ distance_miles`. Ask how
the change in outcome scale changes the slope's units and the residual pattern.

In [ ]:
log_model = smf.ols("log_price ~ distance_miles", data=vienna).fit(cov_type="HC3")
pd.DataFrame(
    {"level_model": simple_model.resid, "log_model": log_model.resid}
).describe().loc[["mean", "std"]].round(3)